In [2]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.5 MB/s eta 0:00:00a 0:00:01


In [3]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from sklearn.preprocessing import MinMaxScaler

In [4]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/shakibulakash/dynamic-graph-datasets/soc-redditHyperlinks-body.tsv
/kaggle/input/datasets/shakibulakash/dynamic-graph-datasets/arxiv_edges.txt/Cit-HepTh.txt
/kaggle/input/datasets/shakibulakash/dynamic-graph-datasets/arxiv_dates.txt/Cit-HepTh-dates.txt
/kaggle/input/datasets/shakibulakash/dynamic-graph-datasets/Enron.txt/Email-Enron.txt


In [5]:
BASE = "/kaggle/input/datasets/shakibulakash/dynamic-graph-datasets/"

In [6]:
reddit_df = pd.read_csv(BASE + "soc-redditHyperlinks-body.tsv", sep='\t')

In [7]:
enron_df = pd.read_csv(BASE + "Enron.txt/Email-Enron.txt",
                       sep='\t', comment='#', header=None)
enron_df.columns = ['src','dst']

In [8]:
arxiv_edges = pd.read_csv(BASE + "arxiv_edges.txt/Cit-HepTh.txt",
                          sep='\t', comment='#', header=None)
arxiv_edges.columns = ['src','dst']

arxiv_dates = pd.read_csv(BASE + "arxiv_dates.txt/Cit-HepTh-dates.txt",
                          sep=r'\s+', comment='#', header=None)

arxiv_dates = arxiv_dates[[0,1]]
arxiv_dates.columns = ['node','date']

In [9]:
reddit_df.head()


,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,POST_ID,TIMESTAMP,LINK_SENTIMENT,PROPERTIES
0,leagueoflegends,teamredditteams,1u4nrps,2013-12-31 16:39:58,1,"345.0,298.0,0.75652173913,0.0173913043478,0.08..."
1,theredlion,soccer,1u4qkd,2013-12-31 18:18:37,-1,"101.0,98.0,0.742574257426,0.019801980198,0.049..."
2,inlandempire,bikela,1u4qlzs,2014-01-01 14:54:35,1,"85.0,85.0,0.752941176471,0.0235294117647,0.082..."
3,nfl,cfb,1u4sjvs,2013-12-31 17:37:55,1,"1124.0,949.0,0.772241992883,0.0017793594306,0...."
4,playmygame,gamedev,1u4w5ss,2014-01-01 02:51:13,1,"715.0,622.0,0.777622377622,0.00699300699301,0...."


In [10]:

enron_df.head()


,src,dst
0,0,1
1,1,0
2,1,2
3,1,3
4,1,4


In [11]:
arxiv_edges.head()


,src,dst
0,1001,9304045
1,1001,9308122
2,1001,9309097
3,1001,9311042
4,1001,9401139


In [12]:
arxiv_dates.head()

,node,date
0,9203201,1992-02-24
1,9203202,1992-03-08
2,9203203,1992-03-03
3,9203204,1992-03-09
4,9203205,1992-03-09


In [13]:
reddit = reddit_df[['SOURCE_SUBREDDIT','TARGET_SUBREDDIT','TIMESTAMP']].copy()

reddit.columns = ['src','dst','time']

# Convert time
reddit['time'] = pd.to_datetime(reddit['time'])

# Remove self-loops
reddit = reddit[reddit['src'] != reddit['dst']]

# Remove duplicates
reddit = reddit.drop_duplicates()

# Sort by time
reddit = reddit.sort_values('time').reset_index(drop=True)

In [14]:
reddit.head()
reddit.shape

(283057, 3)

In [15]:
enron = enron_df.copy()

# Remove self-loops
enron = enron[enron['src'] != enron['dst']]

# Remove duplicates
enron = enron.drop_duplicates()

# Create synthetic time (important)
enron['time'] = pd.to_datetime(np.arange(len(enron)), unit='s')

In [16]:
enron.head()

,src,dst,time
0,0,1,1970-01-01 00:00:00
1,1,0,1970-01-01 00:00:01
2,1,2,1970-01-01 00:00:02
3,1,3,1970-01-01 00:00:03
4,1,4,1970-01-01 00:00:04


In [17]:
arxiv = arxiv_edges.merge(arxiv_dates, left_on='src', right_on='node')

arxiv = arxiv[['src','dst','date']]
arxiv.columns = ['src','dst','time']

# Convert time
arxiv['time'] = pd.to_datetime(arxiv['time'])

# Remove self-loops
arxiv = arxiv[arxiv['src'] != arxiv['dst']]

# Remove duplicates
arxiv = arxiv.drop_duplicates()

# Sort
arxiv = arxiv.sort_values('time').reset_index(drop=True)

In [18]:
arxiv.head()
arxiv.shape

(138577, 3)

In [19]:
print(reddit.columns)
print(enron.columns)
print(arxiv.columns)

Index(['src', 'dst', 'time'], dtype='object')
Index(['src', 'dst', 'time'], dtype='object')
Index(['src', 'dst', 'time'], dtype='object')


In [20]:
def encode_nodes(df):
    
    nodes = pd.concat([df['src'], df['dst']]).unique()
    
    mapping = {node: i for i, node in enumerate(nodes)}
    
    df['src'] = df['src'].map(mapping)
    df['dst'] = df['dst'].map(mapping)
    
    return df, mapping

In [21]:
reddit, reddit_map = encode_nodes(reddit)
enron, enron_map = encode_nodes(enron)
arxiv, arxiv_map = encode_nodes(arxiv)

In [22]:
print(reddit.head())
print(reddit.dtypes)

   src   dst                time
0    0  2093 2013-12-31 16:39:58
1    1   608 2013-12-31 17:37:55
2    2   105 2013-12-31 18:18:37
3    3    10 2013-12-31 18:35:44
4    4   195 2013-12-31 22:27:50
src              int64
dst              int64
time    datetime64[ns]
dtype: object


In [23]:
print(reddit.isnull().sum())

src     0
dst     0
time    0
dtype: int64


In [24]:
def create_slices(df, freq='7D'):
    
    df = df.set_index('time')
    
    slices = []
    
    for _, group in df.groupby(pd.Grouper(freq=freq)):
        if len(group) > 0:
            slices.append(group.reset_index())
    
    return slices

In [25]:
reddit_slices = create_slices(reddit, '7D')

In [26]:
print("Number of time steps:", len(reddit_slices))
print(reddit_slices[0].head())

Number of time steps: 174
                 time  src   dst
0 2013-12-31 16:39:58    0  2093
1 2013-12-31 17:37:55    1   608
2 2013-12-31 18:18:37    2   105
3 2013-12-31 18:35:44    3    10
4 2013-12-31 22:27:50    4   195


In [27]:
for i in range(3):
    print(len(reddit_slices[i]))

654
973
1011


In [28]:
def build_graphs(slices):
    
    graphs = []
    
    for df in slices:
        G = nx.DiGraph()
        
        edges = df[['src','dst']].values
        G.add_edges_from(edges)
        
        graphs.append(G)
    
    return graphs

In [29]:
graphs = build_graphs(reddit_slices)

In [30]:
print("Number of graphs:", len(graphs))
print("First graph nodes:", graphs[0].number_of_nodes())
print("First graph edges:", graphs[0].number_of_edges())

Number of graphs: 174
First graph nodes: 688
First graph edges: 625


In [31]:
print(graphs[10].number_of_nodes(), graphs[10].number_of_edges())
print(graphs[-1].number_of_nodes(), graphs[-1].number_of_edges())

1154 1219
1647 1609


In [32]:
def graph_to_tensor(G):

    nodes = list(G.nodes())
    node_map = {n:i for i,n in enumerate(nodes)}

    # Reindex edges (CRITICAL FIX)
    edges = [(node_map[u], node_map[v]) for u,v in G.edges()]
    
    if len(edges) > 0:
        edge_index = torch.tensor(edges, dtype=torch.long).T
    else:
        edge_index = torch.empty((2,0), dtype=torch.long)

    # Node features
    deg = np.array([G.degree(n) for n in nodes])
    clustering = np.array(list(nx.clustering(G.to_undirected()).values()))

    x = np.vstack([deg, clustering]).T
    x = MinMaxScaler().fit_transform(x)

    x = torch.tensor(x, dtype=torch.float)

    return x, edge_index

In [33]:
data_seq = [graph_to_tensor(g) for g in graphs]

In [34]:
x, edge_index = data_seq[0]

print("Node feature shape:", x.shape)
print("Edge index shape:", edge_index.shape)

Node feature shape: torch.Size([688, 2])
Edge index shape: torch.Size([2, 625])


In [35]:
print(len(data_seq))

174


In [36]:
class TemporalGNN(nn.Module):
    def __init__(self, in_dim, hidden):
        super().__init__()
        
        self.gcn1 = GCNConv(in_dim, hidden)
        self.gcn2 = GCNConv(hidden, hidden)
        
        self.rnn = nn.GRU(hidden, hidden, batch_first=True)
        
        self.fc = nn.Linear(hidden, hidden)

    def forward(self, seq):

        outputs = []

        for x, edge_index in seq:
            if edge_index.shape[1] == 0:
                continue

            h = torch.relu(self.gcn1(x, edge_index))
            h = torch.relu(self.gcn2(h, edge_index))

            outputs.append(h.unsqueeze(0))

        H = torch.cat(outputs, dim=0)   # (time, nodes, dim)
        H = H.transpose(0,1)            # (nodes, time, dim)

        out, _ = self.rnn(H)

        out = self.fc(out[:, -1, :])    # last time step

        return out

In [37]:
model = TemporalGNN(in_dim=2, hidden=64)

In [38]:
print(model)

TemporalGNN(
  (gcn1): GCNConv(2, 64)
  (gcn2): GCNConv(64, 64)
  (rnn): GRU(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=64, bias=True)
)


In [39]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [40]:
max_nodes = max([x.shape[0] for x, _ in data_seq])
print("Max nodes:", max_nodes)

Max nodes: 2199


In [41]:
def pad_tensor(x, max_nodes):
    
    n, d = x.shape
    
    if n < max_nodes:
        pad = torch.zeros((max_nodes - n, d))
        x = torch.cat([x, pad], dim=0)
    
    return x

In [42]:
padded_seq = []

for x, edge_index in data_seq:
    x_padded = pad_tensor(x, max_nodes)
    padded_seq.append((x_padded, edge_index))

In [43]:
for epoch in range(10):

    optimizer.zero_grad()

    pred = model(padded_seq[:-1])
    target = model(padded_seq[1:]).detach()

    loss = loss_fn(pred, target)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}: Loss = {loss.item()}")

Epoch 0: Loss = 7.5088605626660865e-06
Epoch 1: Loss = 5.682476967194816e-06
Epoch 2: Loss = 4.281724613974802e-06
Epoch 3: Loss = 3.215566948711057e-06
Epoch 4: Loss = 2.420371174594038e-06
Epoch 5: Loss = 1.8362865148446872e-06
Epoch 6: Loss = 1.4178750689097797e-06
Epoch 7: Loss = 1.1281232445981004e-06
Epoch 8: Loss = 9.335943218502507e-07
Epoch 9: Loss = 8.037034149310784e-07


In [44]:
def constraint_layout(G, prev_pos=None, alpha=0.7, beta=0.2):

    # Base layout
    pos = nx.spring_layout(G, pos=prev_pos, iterations=15)

    if prev_pos is None:
        return pos

    new_pos = {}

    for node in pos:

        if node in prev_pos:

            # 🔹 Stability constraint
            stable = alpha * prev_pos[node] + (1 - alpha) * pos[node]

            # 🔹 Smoothness (neighbor influence)
            neighbors = list(G.neighbors(node))

            if len(neighbors) > 0:
                valid_neighbors = [pos[n] for n in neighbors if n in pos]

                if len(valid_neighbors) > 0:
                    avg = np.mean(valid_neighbors, axis=0)
                    smooth = (1 - beta) * stable + beta * avg
                    new_pos[node] = smooth
                else:
                    new_pos[node] = stable
            else:
                new_pos[node] = stable

        else:
            # New node
            new_pos[node] = pos[node]

    return new_pos

In [45]:
layouts = []
prev_pos = None

graphs_small = graphs[::2]

for G in graphs_small:

    pos = nx.spring_layout(G, pos=prev_pos, iterations=30)

    new_pos = {}

    for node in pos:

        if prev_pos is not None and node in prev_pos:

            
            alpha = 0.6
            beta = 0.3

            # stability
            stable = alpha * np.array(prev_pos[node]) + (1 - alpha) * np.array(pos[node])

            # smoothing with neighbors
            neighbors = list(G.neighbors(node))

            if len(neighbors) > 0:
                neighbor_pos = [pos[n] for n in neighbors if n in pos]

                if len(neighbor_pos) > 0:
                    avg = np.mean(neighbor_pos, axis=0)
                    new_pos[node] = (1 - beta) * stable + beta * avg
                else:
                    new_pos[node] = stable
            else:
                new_pos[node] = stable

        else:
            # new node
            new_pos[node] = pos[node]

    layouts.append(new_pos)
    prev_pos = new_pos

In [46]:
print(len(layouts))

87


In [47]:
baseline_static = [nx.spring_layout(G, iterations=30) for G in graphs_small]

In [48]:
baseline_dynamic = []
prev = None

for G in graphs_small:
    pos = nx.spring_layout(G, pos=prev, iterations=30)
    baseline_dynamic.append(pos)
    prev = pos

In [49]:
def stability(p1, p2):
    nodes = set(p1) & set(p2)
    
    return np.mean([
        np.linalg.norm(np.array(p1[n]) - np.array(p2[n]))
        for n in nodes
    ])

In [50]:
def edge_variance(G, pos):
    lengths = []
    
    for u, v in G.edges():
        if u in pos and v in pos:
            lengths.append(np.linalg.norm(np.array(pos[u]) - np.array(pos[v])))
    
    return np.var(lengths) if len(lengths) > 0 else 0

In [60]:
def evaluate(graphs, layouts):
    
    stab_list = []
    var_list = []
    
    for i in range(1, len(graphs)):
        
        stab_list.append(stability(layouts[i-1], layouts[i]))
        var_list.append(edge_variance(graphs[i], layouts[i]))
    
    return np.mean(stab_list), np.mean(var_list)

In [61]:
print("Proposed:", evaluate(graphs_small, layouts))
print("Dynamic :", evaluate(graphs_small, baseline_dynamic))
print("Static  :", evaluate(graphs_small, baseline_static))

Proposed: (np.float64(0.19805440748665434), np.float64(0.014827301522413246))
Dynamic : (np.float64(0.39339574660363935), np.float64(0.003119645449545783))
Static  : (np.float64(0.672591437253215), np.float64(0.003185252606401936))


In [62]:
def node_movement(p1, p2):
    nodes = set(p1) & set(p2)
    
    movements = [
        np.linalg.norm(np.array(p1[n]) - np.array(p2[n]))
        for n in nodes
    ]
    
    return np.sum(movements)   

In [63]:
def stability(p1, p2):
    nodes = set(p1) & set(p2)
    
    return np.mean([
        np.linalg.norm(np.array(p1[n]) - np.array(p2[n]))
        for n in nodes
    ])

In [64]:
def evaluate_full(graphs, layouts):
    
    stab_list = []
    var_list = []
    move_list = []
    
    for i in range(1, len(graphs)):
        
        stab_list.append(stability(layouts[i-1], layouts[i]))
        var_list.append(edge_variance(graphs[i], layouts[i]))
        move_list.append(node_movement(layouts[i-1], layouts[i]))
    
    return np.mean(stab_list), np.mean(var_list), np.mean(move_list)

In [65]:
print("Proposed:", evaluate_full(graphs_small, layouts))

Proposed: (np.float64(0.19805440748665434), np.float64(0.014827301522413246), np.float64(124.77480805702878))


In [67]:


layouts_pred = []
prev_pos = None

graphs_small = graphs[::2]

# make sure prediction exists
with torch.no_grad():
    pred_embeddings = model(padded_seq)

pred_np = pred_embeddings.numpy()

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
pred_2d = scaler.fit_transform(pred_np[:, :2])

lambda_pred = 0.2

for G in graphs_small:

    pos = nx.spring_layout(G, pos=prev_pos, iterations=30)

    new_pos = {}

    for node in pos:

        base = np.array(pos[node])

        if prev_pos is not None and node in prev_pos:
            base = 0.6 * np.array(prev_pos[node]) + 0.4 * base

        # 🔥 prediction influence
        if node < len(pred_2d):
            new_pos[node] = base + lambda_pred * pred_2d[node]
        else:
            new_pos[node] = base

    layouts_pred.append(new_pos)
    prev_pos = new_pos

print("layouts_pred created:", len(layouts_pred))

layouts_pred created: 87


In [68]:
print("Original Proposed:", evaluate_full(graphs_small, layouts))
print("Prediction Layout:", evaluate_full(graphs_small, layouts_pred))

Original Proposed: (np.float64(0.19805440748665434), np.float64(0.014827301522413246), np.float64(124.77480805702878))
Prediction Layout: (np.float64(0.15545728554957883), np.float64(0.02374864903053234), np.float64(97.79259797863354))
